# Mediterranean Sea Surface Salinity — Climatology, Anomalies & Extremes (1987–2025)

**Workflow**
1. Open the Mediterranean Sea Physics Reanalysis (`cmems_mod_med_phy-sal_my_4.2km_P1M-m`) — monthly means, lazy (dask).
2. Compute the **yearly mean salinity** at every pixel and explore it with a year slider.
3. Compute the **long-term climatological mean** at every pixel (mean over 1987–2025).
4. Compute the **monthly salinity anomaly** = monthly value − pixel climatological mean.
5. Visualise the monthly anomaly with an interactive time slider showing the current month.
6. At every pixel, find the **minimum** and **maximum** of the anomaly time-series.
7. Plot spatial maps of the min-anomaly and max-anomaly fields.

> **Credentials**: requires a `~/.netrc` entry for `auth.marine.copernicus.eu`.

In [1]:
import cmocean
import copernicusmarine
import hvplot.xarray
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Mediterranean bounding box
MED_BBOX = dict(
    minimum_longitude=-6,
    maximum_longitude=42,
    minimum_latitude=30,
    maximum_latitude=47,
)

# The reanalysis starts January 1987; request through end of 2025
TIME_START = "1987-01-01"
TIME_END   = "2025-12-31"

## 1 — Open the salinity dataset (lazy)

In [3]:
%%time
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_phy-sal_my_4.2km_P1M-m",
    start_datetime=TIME_START,
    end_datetime=TIME_END,
    **MED_BBOX,
)
ds

INFO - 2026-04-23T17:01:19Z - Selected dataset version: "202511"
INFO - 2026-04-23T17:01:19Z - Selected dataset part: "default"
WARNING - 2026-04-23T17:01:19Z - Some of your subset selection [30.0, 47.0] for the latitude dimension exceed the dataset coordinates [30.1875, 45.97916793823242]
WARNING - 2026-04-23T17:01:19Z - Some of your subset selection [-6.0, 42.0] for the longitude dimension exceed the dataset coordinates [-6.0, 36.29166793823242]


CPU times: user 3.27 s, sys: 973 ms, total: 4.24 s
Wall time: 7.27 s


<xarray.Dataset> Size: 102GB
Dimensions:    (depth: 141, latitude: 380, longitude: 1016, time: 468)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 4kB 1987-01-01 1987-02-01 ... 2025-12-01
Data variables:
    so         (time, depth, latitude, longitude) float32 102GB dask.array<chunksize=(100, 2, 380, 1016), meta=np.ndarray>
Attributes:
    Conventions:               CF-1.0
    bulletin_date:             20241031
    bulletin_type:             interim
    comment:                   Please check in CMEMS catalogue the INFO secti...
    contact:                   servicedesk.cmems@mercator-ocean.eu
    field_type:                monthly_mean_centered_at_time_field
    institution:               Centro Euro-Mediterraneo sui Cambiamenti Clima...
    references:                Escudier, R., Clementi, E., Omar, M., Cipollon...
    source:                    MFS E3R1I
    title:                     Salinity (3D) - Monthly Mean
    copernicusmarine_version:  2.3.0

In [4]:
# Surface salinity: select depth nearest to 0 m — result is (time, latitude, longitude), still lazy
sal = ds["so"].sel(depth=0, method="nearest")
print(f"Salinity shape: {sal.dims} = {dict(zip(sal.dims, sal.shape))}")
sal

Salinity shape: ('time', 'latitude', 'longitude') = {'time': 468, 'latitude': 380, 'longitude': 1016}


<xarray.DataArray 'so' (time: 468, latitude: 380, longitude: 1016)> Size: 723MB
dask.array<getitem, shape=(468, 380, 1016), dtype=float32, chunksize=(100, 380, 1016), chunktype=numpy.ndarray>
Coordinates:
    depth      float32 4B 1.018
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 4kB 1987-01-01 1987-02-01 ... 2025-12-01
Attributes:
    long_name:      salinity
    standard_name:  sea_water_salinity
    units:          0.001
    valid_max:      42.0
    valid_min:      1.0

## 2 — Yearly mean salinity with time slider

Resample monthly → annual, compute (≈ 39 frames × 380 × 1016 ≈ 60 MB), then display with a year scrubber.

In [5]:
%%time
sal_yearly = sal.resample(time="YE").mean().compute()

# Replace the datetime coordinate with plain integer years for a cleaner slider
years = sal_yearly.time.dt.year.values
sal_yearly = (
    sal_yearly
    .assign_coords(year=("time", years))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
sal_yearly.attrs["long_name"] = "Annual Mean Salinity"
sal_yearly.attrs["units"] = "PSU"
print(f"Yearly salinity shape: {sal_yearly.dims} = {dict(zip(sal_yearly.dims, sal_yearly.shape))}")
sal_yearly

Yearly salinity shape: ('year', 'latitude', 'longitude') = {'year': 39, 'latitude': 380, 'longitude': 1016}
CPU times: user 15.6 s, sys: 8.16 s, total: 23.8 s
Wall time: 8.82 s


<xarray.DataArray 'so' (year: 39, latitude: 380, longitude: 1016)> Size: 60MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]],
      shape=(39, 380, 1016), dtype=float32)
Coordinates:
    depth      float32 4B 1.018
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * year       (year) int64 312B 1987 1988 1989 1990 ... 2022 2023 2024 2025
Attributes:
    long_name:      Annual Mean Salinity
    standard_name:  sea_water_salinity
    units:          PSU
    valid_max:      42.0
    valid_min:      1.0

In [6]:
sal_yearly.hvplot(
    x="longitude",
    y="latitude",
    groupby="year",
    rasterize=True,
    geo=True,
    cmap=cmocean.cm.haline,
    clim=(float(sal_yearly.min()), float(sal_yearly.max())),
    clabel="Salinity (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title="Mediterranean Annual Mean Salinity — use slider to step through years",
    fontscale=1.2,
    widget_type="scrubber",
    widget_location="bottom",
)

Column
    [0] HoloViews(DynamicMap, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber')
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=38, width=550)

## 3 — Long-term climatological mean

Average every pixel over the full record (1987–2025). This is the reference field for anomaly computation.

In [7]:
%%time
sal_clim = sal.mean(dim="time").compute()
sal_clim.attrs["long_name"] = f"Climatological Mean Salinity ({TIME_START[:4]}\u2013{TIME_END[:4]})"
sal_clim.attrs["units"] = "PSU"
print(f"Salinity clim range: {float(sal_clim.min()):.2f} \u2013 {float(sal_clim.max()):.2f} PSU")
sal_clim

Salinity clim range: 31.29 – 40.07 PSU
CPU times: user 11 s, sys: 6.27 s, total: 17.2 s
Wall time: 6.86 s


<xarray.DataArray 'so' (latitude: 380, longitude: 1016)> Size: 2MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(380, 1016), dtype=float32)
Coordinates:
    depth      float32 4B 1.018
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
Attributes:
    long_name:  Climatological Mean Salinity (1987–2025)
    units:      PSU

In [8]:
sal_clim.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap=cmocean.cm.haline,
    clabel="Salinity (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Climatological Mean Salinity ({TIME_START[:4]}–{TIME_END[:4]})",
    fontscale=1.2,
)

:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (Climatological Mean Salinity (1987–2025))

## 4 — Monthly salinity anomaly

Anomaly = monthly salinity − pixel climatological mean.

The subtraction broadcasts `sal_clim` (2-D) over the time axis of `sal` (3-D). The result remains a lazy dask array — no data is loaded into RAM until we ask for an explicit reduction.

In [9]:
sal_anom = sal - sal_clim
sal_anom.attrs["long_name"] = "Salinity Anomaly"
sal_anom.attrs["units"] = "PSU"
print(f"Anomaly shape: {sal_anom.dims} = {dict(zip(sal_anom.dims, sal_anom.shape))}")
sal_anom

Anomaly shape: ('time', 'latitude', 'longitude') = {'time': 468, 'latitude': 380, 'longitude': 1016}


<xarray.DataArray 'so' (time: 468, latitude: 380, longitude: 1016)> Size: 723MB
dask.array<sub, shape=(468, 380, 1016), dtype=float32, chunksize=(100, 380, 1016), chunktype=numpy.ndarray>
Coordinates:
    depth      float32 4B 1.018
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 4kB 1987-01-01 1987-02-01 ... 2025-12-01
Attributes:
    long_name:  Salinity Anomaly
    units:      PSU

## 4b — Monthly salinity anomaly map with time slider

In [10]:
import pandas as pd
import panel as pn

pn.extension()

# Load the full monthly anomaly into memory — needed for a responsive time slider
sal_anom_loaded = sal_anom.compute()

# Symmetric color limits so blue/red are balanced around zero
anom_max = float(max(abs(sal_anom_loaded.min()), abs(sal_anom_loaded.max())))
times = sal_anom_loaded.time.values

# Integer-index Player widget (play/pause/step controls + scrubber)
player = pn.widgets.Player(
    start=0, end=len(times) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_sal_frame(idx):
    label = pd.Timestamp(times[idx]).strftime("%B %Y")
    return sal_anom_loaded.isel(time=idx).hvplot(
        x="longitude",
        y="latitude",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_max, anom_max),
        clabel="Salinity Anomaly (PSU)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Monthly Salinity Anomaly \u2014 {label}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_sal_frame, player), player)

Column
    [0] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [1] Player(end=467, loop_policy='loop', width=800)

## 4c — Annual salinity anomaly with year slider

Annual anomaly = yearly mean salinity (from section 2) − long-term climatological mean (from section 3).

In [11]:
import pandas as pd
import panel as pn

pn.extension()

# Annual anomaly: yearly mean − climatological mean (broadcasts over year dimension)
sal_annual_anom = sal_yearly - sal_clim
sal_annual_anom.attrs["long_name"] = "Annual Salinity Anomaly"
sal_annual_anom.attrs["units"] = "PSU"

# Symmetric color limits
anom_yearly_max = float(max(abs(sal_annual_anom.min()), abs(sal_annual_anom.max())))
years_list = sal_annual_anom.year.values

player_yearly = pn.widgets.Player(
    start=0, end=len(years_list) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_sal_annual_frame(idx):
    yr = int(years_list[idx])
    return sal_annual_anom.isel(year=idx).hvplot(
        x="longitude",
        y="latitude",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_yearly_max, anom_yearly_max),
        clabel="Salinity Anomaly (PSU)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Annual Salinity Anomaly — {yr}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_sal_annual_frame, player_yearly), player_yearly)

Column
    [0] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [1] Player(end=38, loop_policy='loop', width=800)

## 4d — Export annual salinity anomaly animation as MP4

Render one frame per year (1987–2025) and save as a downloadable MP4. Requires `ffmpeg`.

In [12]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

VIDEO_PATH_ANNUAL = "med_sal_annual_anomaly.mp4"

lons = sal_annual_anom.longitude.values
lats = sal_annual_anom.latitude.values

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor("#c8e6f5")

data0 = sal_annual_anom.isel(year=0).values
img = ax.pcolormesh(
    lons, lats, data0,
    cmap="RdBu_r",
    vmin=-anom_yearly_max, vmax=anom_yearly_max,
    shading="auto",
)
fig.colorbar(img, ax=ax, label="Salinity Anomaly (PSU)", fraction=0.03, pad=0.02)
title = ax.set_title(
    f"Mediterranean Annual Salinity Anomaly — {int(years_list[0])}",
    fontsize=14,
)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.tight_layout()

def update_annual(frame_idx):
    data = sal_annual_anom.isel(year=frame_idx).values
    img.set_array(data.ravel())
    title.set_text(f"Mediterranean Annual Salinity Anomaly — {int(years_list[frame_idx])}")
    return img, title

ani = animation.FuncAnimation(
    fig,
    update_annual,
    frames=len(years_list),
    interval=400,
    blit=False,
)

ani.save(VIDEO_PATH_ANNUAL, writer="ffmpeg", fps=3, dpi=120)
plt.close(fig)
print(f"Video saved → {VIDEO_PATH_ANNUAL}")

Video saved → med_sal_annual_anomaly.mp4


In [13]:
from IPython.display import FileLink, display
display(FileLink(VIDEO_PATH_ANNUAL, result_html_prefix="Download video: "))

/home/jovyan/MAGICA-School-2026/Group Project/med_sal_annual_anomaly.mp4

## 5 — Pixel-wise minimum and maximum anomaly

Collapse the time dimension by taking `min` and `max`. Both reductions are fused into a single dask graph pass over the data.

In [ ]:
%%time
import dask

# Compute both reductions in one scheduler call to share the I/O
sal_anom_min, sal_anom_max = dask.compute(
    sal_anom.min(dim="time"),
    sal_anom.max(dim="time"),
)

sal_anom_min.attrs["long_name"] = "Minimum Salinity Anomaly"
sal_anom_min.attrs["units"] = "PSU"
sal_anom_max.attrs["long_name"] = "Maximum Salinity Anomaly"
sal_anom_max.attrs["units"] = "PSU"

print(f"Min anomaly range : {float(sal_anom_min.min()):.3f} \u2013 {float(sal_anom_min.max()):.3f} PSU")
print(f"Max anomaly range : {float(sal_anom_max.min()):.3f} \u2013 {float(sal_anom_max.max()):.3f} PSU")

## 6 — Spatial maps of min and max anomaly

A diverging colormap (`RdBu_r`) centred on zero makes fresher/saltier departures immediately readable.

In [ ]:
# Shared symmetric color limits so both maps are directly comparable
anom_abs_max = max(
    abs(float(sal_anom_min.min())),
    abs(float(sal_anom_max.max())),
)
clim_anom = (-anom_abs_max, anom_abs_max)
print(f"Symmetric color range: {clim_anom[0]:.3f} \u2013 {clim_anom[1]:.3f} PSU")

In [ ]:
plot_min = sal_anom_min.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Minimum Salinity Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_min

In [ ]:
plot_max = sal_anom_max.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Maximum Salinity Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_max

## 7 — Side-by-side comparison

In [ ]:
plot_min_small = sal_anom_min.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Min Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

plot_max_small = sal_anom_max.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Max Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

(plot_min_small + plot_max_small).cols(2)